# 🚀 Multi-Agent Memory Conflict Resolution - Colab Runner

Notebook này chạy toàn bộ benchmark với LLM agents trên Google Colab GPU.

**Lưu ý:**
- Colab có time limit ~12h, nên nên chạy `--max-scenarios 4` trước để test
- Dùng model nhỏ (1.5B) để tránh OOM trên T4 GPU
- Data được lưu trên Google Drive để persist qua sessions

In [ ]:
# 1. Mount Google Drive để lưu data và results
from google.colab import drive
drive.mount('/content/drive')

print('Drive mounted at /content/drive')

# Tạo thư mục project trên Drive
PROJECT_DIR = '/content/drive/MyDrive/ProjectMemColab'
!mkdir -p "$PROJECT_DIR"
print(f'Project directory: {PROJECT_DIR}')

## 2. Upload Code hoặc Clone từ GitHub

**Option A: Nếu bạn đã push code lên GitHub** (khuyến nghị):

In [ ]:
# 2. Clone hoặc Update code từ GitHub
import os
import subprocess

GITHUB_REPO = input('Enter GitHub repo URL (để trống nếu code đã có trên Drive): ').strip()

os.chdir(PROJECT_DIR)

if GITHUB_REPO:
    if os.path.exists('.git'):
        print('Repository đã tồn tại, đang pull updates...')
        subprocess.run(['git', 'pull'], check=False)
        print('Pull complete!')
    else:
        print('Cloning repository...')
        subprocess.run(['git', 'clone', GITHUB_REPO, '.'], check=False)
        print('Clone complete!')
else:
    # Kiểm tra xem có code sẵn không
    if os.path.exists('main.py'):
        print('Code có sẵn trên Drive - không cần clone')
    else:
        print('WARNING: Không tìm thấy main.py. Vui lòng upload code hoặc nhập GitHub URL!')

print(f'Working directory: {os.getcwd()}')
!ls -la

**Option B: Upload trực tiếp từ local (nếu chưa có GitHub)**

In [ ]:
from google.colab import files
import shutil
import os

# Nếu thư mục project trống, upload code
if not os.path.exists('/content/drive/MyDrive/ProjectMemColab/main.py'):
    print('Vui lòng upload toàn bộ project folder (zip hoặc từng file)')
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        # Giải nén nếu là zip
        if filename.endswith('.zip'):
            !unzip "$filename" -d "$PROJECT_DIR"
        else:
            shutil.copy2(filename, os.path.join(PROJECT_DIR, filename))
    
    print('Upload complete!')
%cd "$PROJECT_DIR"

## 3. Cài đặt Dependencies

Cài đặt tất cả packages từ requirements.txt

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed!')

## 4. Download Benchmark Data (nếu chưa có)

Data được lưu trong thư mục `data/raw/`. MAB dataset sẽ tự động download từ HuggingFace lần đầu chạy.

MemAB data cần được upload thủ công lên Drive nếu bạn có file parquet.

In [ ]:
import os

# Kiểm tra data
print('=== Checking data directories ===')
if os.path.exists('data/raw/memab'):
    memab_files = os.listdir('data/raw/memab')
    print(f'MemAB: {memab_files}')
else:
    print('MemAB: NOT FOUND - vui lòng upload Conflict_Resolution.parquet vào data/raw/memab/')

if os.path.exists('data/raw/longmemeval'):
    longmem_files = os.listdir('data/raw/longmemeval')
    print(f'LongMemEval: {len(longmem_files)} files')
else:
    print('LongMemEval: NOT FOUND')

print('\nNếu thiếu data, tải từ HuggingFace sẽ xảy ra khi chạy benchmark')

## 5. Cấu hình Model

Colab T4 GPU (12GB) nên dùng model nhỏ:
- **Qwen/Qwen2.5-1.5B-Instruct** (~3GB với 4bit quantization)
- **microsoft/phi-2** (2.7B, rất nhẹ)

**Không dùng model 7B** - sẽ OOM.

In [ ]:
import os

# Model configuration - thay đổi tại đây nếu cần
AGENT1_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
AGENT2_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

# Số scenarios để chạy (giảm nếu cần)
MAX_SCENARIOS = 8  # Thử với 2-4 trước, sau đó tăng lên 8

# Benchmark selection
BENCHMARK = 'real_conflicts'  # options: memae, mab_conflict, real_conflicts

# Lưu config
with open('colab_config.txt', 'w') as f:
    f.write(f'AGENT1_MODEL={AGENT1_MODEL}\n')
    f.write(f'AGENT2_MODEL={AGENT2_MODEL}\n')
    f.write(f'MAX_SCENARIOS={MAX_SCENARIOS}\n')
    f.write(f'BENCHMARK={BENCHMARK}\n')

print(f'Config: {AGENT1_MODEL} | {AGENT2_MODEL} | scenarios={MAX_SCENARIOS} | benchmark={BENCHMARK}')

## 6. Chạy Benchmark

Lệnh sẽ chạy với các settings đã cấu hình. Kết quả lưu vào `reports/`.

⚠️ **Time estimate:**
- 4 scenarios với model 1.5B: ~2-4 giờ
- 8 scenarios: ~4-8 giờ

Colab có thể ngắt sau 12h. Nếu bị ngắt, có thể resume từ reports cũ bằng cách merge.

In [ ]:
import subprocess
import sys
from datetime import datetime

# Build command
cmd = [
    'python', 'main.py',
    '--benchmark', BENCHMARK,
    '--max-scenarios', str(MAX_SCENARIOS),
    '--agent1-model', AGENT1_MODEL,
    '--agent2-model', AGENT2_MODEL,
    '--output-dir', 'reports'
]

print('='*70)
print('STARTING BENCHMARK')
print('='*70)
print('Command:', ' '.join(cmd))
print(f'Start time: {datetime.now()}')
print('='*70)

# Run with real-time output
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

# Stream output
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

process.wait()

print('\n' + '='*70)
print(f'FINISHED with exit code: {process.returncode}')
print(f'End time: {datetime.now()}')

## 7. Đọc Kết Quả

Sau khi chạy xong, đọc báo cáo từ file JSON.

In [ ]:
import json
import os

report_files = {
    'MemAE': 'reports/memae_report.json',
    'MAB': 'reports/mab_report.json',
    'Summary': 'reports/summary.json'
}

for name, path in report_files.items():
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f'\n=== {name} ===')
        print(json.dumps(data, indent=2)[:2000])
        if name == 'Summary':
            print('\nFull summary saved to:', path)
    else:
        print(f'\n{name}: NOT FOUND (chưa chạy xong?)')

## 8. Copy Reports về Local (tùy chọn)

Tải báo cáo về máy để phân tích.

In [ ]:
from google.colab import files
import shutil

# Zip reports folder
shutil.make_archive('reports', 'zip', 'reports')

# Download
files.download('reports.zip')

print('Reports downloaded as reports.zip')

---

## Troubleshooting

**CUDA Out of Memory:**
- Giảm `MAX_SCENARIOS` xuống 2
- Dùng model nhỏ hơn: `microsoft/phi-2`
- Thêm quantization trong code agent (nếu có)

**Time Limit Exceeded:**
- Chia nhỏ: chạy `--benchmark memae` rồi `--benchmark mab_conflict` riêng

**Data not found:**
- Upload MemAB parquet vào `data/raw/memab/` trên Drive
- MAB dataset tự động download từ HuggingFace